# Bayan · local wake-word model (Colab, GPU) — trains "Bayan"

Produces a **local ONNX wake detector** to REPLACE browser SpeechRecognition. Trains an
openWakeWord-style classifier for **Bayan / بيان / يا بيان**, exports **ONNX** for the browser
(onnxruntime-web), and includes threshold tuning + false-activation testing.

**Runtime:** Colab → Runtime → Change runtime type → **GPU (T4)**.

**Phased plan (important):** the FIRST model is trained on **Egyptian only** (pilot of ~5 consenting
speakers). Saudi / Arabic-English / Indian / Pakistani / Filipino recordings are collected with the
SAME recorder tool later and added incrementally — you retrain with a bumped version. You do NOT
need all accents before the first Egyptian model.

> Note on TTS: openWakeWord bootstraps POSITIVE training clips with a synthetic-speech generator
> (piper-sample-generator). That TTS is used ONLY to synthesize *wake-word training audio for the
> classifier* — it is **not** Bayan's spoken voice and is unrelated to the NAMAA voice decision.

## Recordings (produced by `tools/bayan-wake-recorder.html`)
The recorder outputs a **versioned** `wake_data_<version>.zip` already in the right format
(mono WAV, 16-bit, 16 kHz) and folder layout. You just upload it below.

```
wake_data/
  positive/egyptian/*.wav   positive/saudi/*.wav   positive/ar_en/*.wav
  positive/indian/*.wav     positive/pakistani/*.wav  positive/filipino/*.wav
  hard_negative/*.wav
  recording_manifest.json   # datasetVersion, scope, per-accent counts, speakers, sha256s
```

**Pilot 1 (accepted):** the v1 dataset is **24 Egyptian positives from a single speaker (quiet, one mic),
0 user hard-negatives** — deliberately small. Synthetic augmentation + openWakeWord's auto-downloaded
generic negatives carry the rest. This is enough to train and deploy a first model; its real quality is
proven by the DEPLOYED benchmark, not by these counts. Expand (more speakers / noise / accents) ONLY if
the deployed benchmark shows weakness. Do NOT use celebrity / scraped / TV / social audio or unconsented voices.

**Honesty note:** with one speaker, detection on that speaker will look high but does NOT prove
cross-speaker performance; and false-activations/hour must be MEASURED against real long negative audio.

In [ ]:
# 1) Install openWakeWord training stack
%pip -q install openwakeword torch torchaudio onnx onnxruntime scipy soundfile pyyaml
import openwakeword, torch
print('openWakeWord', openwakeword.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 1b) Version + scope. The FIRST model is Egyptian-only; add other accents later + retrain (bump versions).
MODEL_VERSION   = 'bayan-wake-v1-egyptian'   # bump for every trained model
DATASET_VERSION = 'v1-egyptian-pilot'        # must match the recorder export you uploaded
TRAIN_ACCENTS   = ['egyptian']               # accents included as positives THIS run
ALL_ACCENTS     = ['egyptian','saudi','ar_en','indian','pakistani','filipino']
print('model', MODEL_VERSION, '| dataset', DATASET_VERSION, '| training accents', TRAIN_ACCENTS)

In [ ]:
# 2) Upload the recorder's wake_data_<version>.zip and unpack
from google.colab import files
import zipfile, glob
up = files.upload()                                   # choose wake_data_<version>.zip
z = [f for f in up if f.endswith('.zip')][0]
zipfile.ZipFile(z).extractall('.')
print('positives:', len(glob.glob('wake_data/positive/**/*.wav', recursive=True)))
print('hard negatives:', len(glob.glob('wake_data/hard_negative/*.wav')))

In [ ]:
# 2b) VALIDATE recordings before training (format + coverage, scoped to TRAIN_ACCENTS). Fails fast.
import glob, os, json, wave
def wav_ok(p):
    try:
        with wave.open(p, 'r') as w:
            return dict(sr=w.getframerate(), ch=w.getnchannels(), bits=w.getsampwidth()*8,
                        secs=round(w.getnframes()/max(1, w.getframerate()), 2))
    except Exception as e:
        return {'error': str(e)}
man_path = 'wake_data/recording_manifest.json'
if os.path.exists(man_path):
    man = json.load(open(man_path))
    print('dataset manifest: version', man.get('datasetVersion'), '| scope', man.get('scope'),
          '| totals', man.get('totals'), '| byAccent', man.get('positivesByAccent'))
    if man.get('datasetVersion') and man['datasetVersion'] != DATASET_VERSION:
        print(f"WARN manifest datasetVersion={man['datasetVersion']} != DATASET_VERSION={DATASET_VERSION}; align the config cell.")
else:
    print('No recording_manifest.json (older export) — continuing on folder scan only.')
def norm(p): return p.replace('\\', '/')
pos_all = glob.glob('wake_data/positive/**/*.wav', recursive=True)
neg = glob.glob('wake_data/hard_negative/*.wav')
train_pos = [p for p in pos_all if any(f'/positive/{a}/' in norm(p) for a in TRAIN_ACCENTS)]
bad_fmt, warns = [], []
for p in train_pos + neg:
    i = wav_ok(p)
    if 'error' in i: bad_fmt.append(f'{p}: {i["error"]}'); continue
    if not (i['sr'] == 16000 and i['ch'] == 1 and i['bits'] == 16):
        bad_fmt.append(f'{p}: got {i["sr"]}Hz/{i["ch"]}ch/{i["bits"]}bit, need 16000/1/16')
    elif not (0.4 <= i['secs'] <= 3.0):
        warns.append(f'{p}: {i["secs"]}s (expect ~1-2s)')
per_accent = {a: len(glob.glob(f'wake_data/positive/{a}/*.wav')) for a in ALL_ACCENTS}
print('positives present by accent:', {k: v for k, v in per_accent.items() if v})
print('training positives (scope', TRAIN_ACCENTS, '):', len(train_pos), '| hard negatives:', len(neg))
if warns:
    print('WARN (accepted):\n  ' + '\n  '.join(warns[:20]) + ('' if len(warns) <= 20 else f'\n  (+{len(warns)-20} more)'))
assert not bad_fmt, 'Bad recordings (must be mono/16-bit/16kHz WAV):\n  ' + '\n  '.join(bad_fmt[:40])
assert train_pos, f'No positives for TRAIN_ACCENTS={TRAIN_ACCENTS}. Record them, or change the scope.'
if len(train_pos) < 30: print(f'WARN only {len(train_pos)} scoped positives (Egyptian pilot target ~30 from ~5 speakers; synthetic augmentation helps but real clips matter).')
if len(neg) < 50: print(f'WARN only {len(neg)} hard negatives (pilot target ~50; generic negatives are added at train time).')
print('Format validation passed.')

In [ ]:
# 3) Synthetic positive augmentation via piper-sample-generator (openWakeWord's documented flow).
#    Makes MANY 'Bayan' pronunciations to augment your REAL positives — NOT Bayan's voice.
#    >>> VERSION FLAG: piper-sample-generator + openWakeWord train.py change CLIs between commits;
#    pin a commit and confirm flag names against that commit's README before a long run.
import os, subprocess, glob
os.makedirs('synthetic_positive', exist_ok=True)
if not os.path.isdir('piper-sample-generator'):
    subprocess.run(['git','clone','-q','https://github.com/rhasspy/piper-sample-generator.git'], check=True)
CKPT = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(CKPT):
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    subprocess.run(['wget','-q','-O',CKPT,
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'], check=False)
TARGET_PHRASES = ['Bayan','بيان','يا بيان','Bayaan','Beyan']
N_PER_PHRASE = 500     # ~2500 synthetic positives; raise for a stronger model
for i, phrase in enumerate(TARGET_PHRASES):
    outdir = f'synthetic_positive/p{i}'; os.makedirs(outdir, exist_ok=True)
    r = subprocess.run(['python3','piper-sample-generator/generate_samples.py', phrase,
        '--model', CKPT, '--max-samples', str(N_PER_PHRASE), '--output-dir', outdir, '--batch-size','50'],
        capture_output=True, text=True)
    print(phrase, '->', len(glob.glob(outdir+'/*.wav')), 'clips', ('' if r.returncode==0 else '\n  STDERR: '+r.stderr[-400:]))
print('synthetic positives total:', len(glob.glob('synthetic_positive/**/*.wav', recursive=True)))
print('If a phrase produced 0 clips, confirm generate_samples.py flags against the pinned commit README.')

In [ ]:
# 4) Train the VERSIONED model on the SCOPED accents (Egyptian for v1) + synthetic positives.
import os, glob, subprocess, yaml
if not os.path.isdir('openWakeWord'):
    subprocess.run(['git','clone','-q','https://github.com/dscripka/openWakeWord.git'], check=True)
os.makedirs('bayan_out', exist_ok=True)
scoped_pos = [f'wake_data/positive/{a}' for a in TRAIN_ACCENTS if os.path.isdir(f'wake_data/positive/{a}')]
positive_dirs = scoped_pos + sorted(glob.glob('synthetic_positive/p*'))
neg_dir = 'wake_data/hard_negative'
negative_dirs = [neg_dir] if glob.glob(neg_dir + '/*.wav') else []   # v1 pilot has 0 user negatives; openWakeWord still auto-downloads a large generic negative corpus
config = {
    'model_name': MODEL_VERSION,
    'target_phrase': ['Bayan','بيان','يا بيان'],
    'model_type': 'dnn',
    'output_dir': os.path.abspath('bayan_out'),
    'n_samples': 20000, 'n_samples_val': 2000,
    'steps': 50000, 'target_accuracy': 0.7, 'target_recall': 0.5,
    'sample_rate': 16000,
    'custom_positive_clips_dirs': [os.path.abspath(d) for d in positive_dirs],
    'custom_negative_clips_dirs': [os.path.abspath(d) for d in negative_dirs],
    'augmentation_rounds': 1, 'tflite': True, 'onnx': True,
}
with open('bayan_config.yaml','w') as f: yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
print('positive dirs:', positive_dirs)
print('wrote bayan_config.yaml:\n', open('bayan_config.yaml').read())
# >>> VERSION FLAG: confirm train.py flags against the pinned openWakeWord commit README.
cmd = ['python3','openWakeWord/openwakeword/train.py','--training_config','bayan_config.yaml',
       '--generate_clips','--augment_clips','--train_model']
print('\nRunning:', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-3000:]); print('STDERR tail:\n', r.stderr[-2000:])
print('\nExit', r.returncode, '| onnx:', glob.glob('bayan_out/**/*.onnx', recursive=True))
assert r.returncode == 0, 'Training failed — read STDERR; if a CLI-flag error, confirm train.py flags against the pinned openWakeWord commit README.'

In [ ]:
# 5) HONEST evaluation — measure real numbers; do not assume them.
#    This v1 pilot is SINGLE-SPEAKER, so treat results carefully:
#    - Hold out a few of AE's clips (never train on them) for an on-speaker detection sanity check.
#      That number shows the model learned the phrase; it does NOT prove other speakers will trigger it.
#    - False-activations/hour: play LONG real negative audio (meetings, TV/YouTube Arabic incl. بيان
#      sentences, music, background talk) through the detector and COUNT activations per hour.
#    - Latency: measure P95 time from end-of-phrase to wake event.
#    - Require consecutiveFrames (2-3) over threshold + cooldown to kill single-frame spikes.
#    Fill quiet/noisy detection %, false-rejection %, false-activations/hour, P95 latency into METRICS
#    in the export cell. Leave any you did not measure as null — never invent a number.
print('Measure on-speaker detection + false-activations/hour + latency, then set METRICS below. '
      'Single-speaker pilot: cross-speaker quality is proven only by the deployed benchmark.')

In [ ]:
# 6) Export ONNX + VERSIONED wake_meta.json (ties this model to its exact dataset).
import shutil, json, hashlib, os, glob
os.makedirs('bayan-wake-out', exist_ok=True)
cands = glob.glob('bayan_out/**/*.onnx', recursive=True)
assert cands, 'No .onnx under bayan_out — training must succeed first.'
onnx_src = max(cands, key=os.path.getsize)
shutil.copy(onnx_src, 'bayan-wake-out/bayan_wake.onnx')
onnx_sha = hashlib.sha256(open('bayan-wake-out/bayan_wake.onnx','rb').read()).hexdigest()
ds_sha = None
if os.path.exists('wake_data/recording_manifest.json'):
    raw = open('wake_data/recording_manifest.json','rb').read(); ds_sha = hashlib.sha256(raw).hexdigest()
per_accent = {a: len(glob.glob(f'wake_data/positive/{a}/*.wav')) for a in ALL_ACCENTS}
n_real = sum(len(glob.glob(f'wake_data/positive/{a}/*.wav')) for a in TRAIN_ACCENTS)
n_syn  = len(glob.glob('synthetic_positive/**/*.wav', recursive=True))
# Fill from cell 5 before requesting review:
METRICS = {'quietDetection': None, 'noisyDetection': None, 'falseRejection': None,
           'falseActivationsPerHour': None, 'p95LatencyMs': None}
meta = {
  'modelVersion': MODEL_VERSION, 'datasetVersion': DATASET_VERSION, 'trainAccents': TRAIN_ACCENTS,
  'wakePhrases': ['Bayan','بيان','يا بيان','Bayaan','Beyan'],
  'sampleRate': 16000, 'frameMs': 80, 'threshold': 0.5, 'consecutiveFrames': 2, 'cooldownMs': 2000, 'opset': 17,
  'onnxSource': os.path.basename(onnx_src), 'onnxSha256': onnx_sha, 'datasetManifestSha256': ds_sha,
  'trainPositivesReal': n_real, 'trainPositivesSynthetic': n_syn,
  'positivesByAccentInZip': {k: v for k, v in per_accent.items() if v},
  'allAccentsPlanned': ALL_ACCENTS, 'metrics': METRICS,
}
open('bayan-wake-out/wake_meta.json','w').write(json.dumps(meta, ensure_ascii=False, indent=2))
print(json.dumps(meta, ensure_ascii=False, indent=2))
zip_name = f'bayan-wake-model_{MODEL_VERSION}'
shutil.make_archive(zip_name, 'zip', 'bayan-wake-out')
from google.colab import files as F; F.download(zip_name + '.zip')
print('Deliver', zip_name + '.zip', '(bayan_wake.onnx + wake_meta.json). Fill metrics before review. Do NOT enable BAYAN_WAKE_ONNX_ENABLED until reviewed.')

## Browser integration (drop-in, after the model passes review)
1. Commit `bayan_wake.onnx` under a **versioned** path `frontend/public/wake/<modelVersion>/`; bundle
   **onnxruntime-web** locally (pinned; no CDN). Keep `wake_meta.json` beside it so the runtime knows
   which `modelVersion` / `datasetVersion` is live.
2. Add an `OnnxWakeDetector implements WakeDetector` in `frontend/src/bayan/wake.ts`:
   Microphone → **AudioWorklet** (16 kHz frames) → **Web Worker** running onnxruntime-web on
   `bayan_wake.onnx` → require `consecutiveFrames` positives over `threshold` → emit one wake event →
   apply `cooldownMs`. Runs off the main thread; pauses while Bayan speaks/records.
3. Select `OnnxWakeDetector` when `BAYAN_WAKE_ONNX_ENABLED=true` and the model is present, else fall
   back. **Retire `WebSpeechWakeDetector`** as production once ONNX detection meets targets for the
   piloted accent(s).
4. **Incremental expansion:** to add an accent later, record it with the same tool, export a new
   `wake_data_<version>.zip`, set `TRAIN_ACCENTS` (e.g. `['egyptian','saudi']`) + bump
   `MODEL_VERSION`/`DATASET_VERSION`, retrain, and ship the new versioned model.